<a href="https://colab.research.google.com/github/Karna2006/RAG-Basics/blob/main/Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf sentence-transformers langchain \
            langchain-text-splitters langchain-community \
            langchain-core chromadb -q

# Note: we do NOT install langchain-openai
# We wire in Gemini + sentence-transformers manually
# Everything else (LCEL, Chroma, prompts) is pure LangChain

import fitz, numpy as np
import google.generativeai as genai
from google.colab import userdata, files

from langchain_text_splitters   import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents   import Document
from langchain_core.prompts     import ChatPromptTemplate
from langchain_core.runnables   import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers      import SentenceTransformer

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Imports done.")

In [ ]:
#STEP 1 — custom embedding wrapper (bridges sentence-transformers → LangChain)
from langchain_core.embeddings import Embeddings
from typing import List

# LangChain expects an object with .embed_documents() and .embed_query()
# sentence-transformers has .encode() — so we wrap it in one small class.
# This is the ONLY glue code needed to use any embedding model with LangChain.
class STEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text])[0].tolist()

embedding_fn = STEmbeddings()

In [ ]:
#STEP 2 — custom LLM wrapper (bridges Gemini → LangChain)
from langchain_core.language_models import BaseLLM
from langchain_core.outputs import LLMResult, Generation

class GeminiLLM(BaseLLM):
    model_name: str = "gemini-2.5-flash" # Updated model name

    def _generate(self, prompts, stop=None, **kwargs):
        model = genai.GenerativeModel(self.model_name)
        generations = []
        for prompt in prompts:
            resp = model.generate_content(prompt)
            generations.append([Generation(text=resp.text)])
        return LLMResult(generations=generations)

    @property
    def _llm_type(self) -> str:
        return "gemini"

llm = GeminiLLM()

In [ ]:
for m in genai.list_models():
    print(f"{m.name}: {m.supported_generation_methods}")

In [ ]:
#STEP 3 — load PDF + chunk + ingest into Chroma
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Load raw text
doc  = fitz.open(pdf_path)
text = "\n".join(p.get_text() for p in doc)

# LangChain splitter — same RecursiveCharacterTextSplitter you know
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_text(text)

# Wrap chunks in LangChain Document objects (adds metadata support)
docs = [Document(page_content=c, metadata={"source": pdf_path, "chunk_id": i})
        for i, c in enumerate(chunks)]

# Chroma ingests, embeds, and persists in one call
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_fn,
    persist_directory="./chroma_db",   # saves to disk in Colab
    collection_name="my_rag_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"{len(docs)} chunks stored in Chroma.")

In [ ]:
#THE LCEL CHAIN — this is the payoff
# The prompt template — same logic as your naive prompt,
# but now it's a reusable, inspectable, composable object
prompt = ChatPromptTemplate.from_template("""
Answer using ONLY the context below.
If the answer is not in the context, say "I don't have that info."

Context:
{context}

Question: {question}
Answer:""")

# Helper: formats retrieved Document objects into one string
# In naive RAG you did: "\n\n---\n\n".join(chunks)
# This does the same thing but for LangChain Document objects
def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

# ── THE LCEL CHAIN ─────────────────────────────────────────────
#
#   {"context": retriever | format_docs,    ← parallel branch
#    "question": RunnablePassthrough()}     ← passthrough branch
#   | prompt                                ← fills template
#   | llm                                  ← generates answer
#   | StrOutputParser()                    ← extracts text
#
# The | operator is LCEL's pipe. Each step receives the output
# of the previous step. The dict at the start runs TWO things
# in parallel: retriever gets the query, passthrough keeps it.

chain = (
    {
        "context":  retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# That's the entire RAG pipeline. 8 lines.

In [ ]:
answer = chain.invoke("What is the experience of this candidate?")
print(answer)

In [ ]:
#STREAMING — tokens print as they generate (impossible in naive RAG)
for chunk in chain.stream("Summarise the skills of the candidate."):
    print(chunk, end="", flush=True)
print()  # newline at end

In [ ]:
#BATCH — run multiple questions in one call
questions = [
    "What is this document about?",
    "Who are the roles which the candidate is fit for?",
    "What are the qualities of the candidate?",
]
answers = chain.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q}\nA: {a}\n")

In [ ]:
#INSPECT RETRIEVED CHUNKS (debugging — add this when answers are wrong)
# Build a retrieval-only chain to see exactly what was fetched
retrieval_chain = (
    retriever
    | RunnableLambda(lambda docs: [
        {"score_rank": i+1, "text": d.page_content[:120], "meta": d.metadata}
        for i, d in enumerate(docs)
    ])
)
chunks_retrieved = retrieval_chain.invoke("What is the refund policy?")
for c in chunks_retrieved:
    print(f"[{c['score_rank']}] {c['text']}...")

# If the retrieved chunks look wrong → go back and tune chunk_size
# If chunks look right but answer is wrong → tune the prompt

In [ ]:
#RELOAD FROM DISK (don't re-embed every session)
# Next time you open Colab, load the persisted Chroma DB
# instead of re-embedding your entire PDF — saves time + API calls
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_fn,
    collection_name="my_rag_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# Then rebuild the chain with the new retriever — same 8 lines as before

In [23]:
import os

# Create the dochat directory and its subdirectories
# This ensures the project structure is in place before creating files
if not os.path.exists('dochat/chroma_db'):
    os.makedirs('dochat/chroma_db')
    print("Created directory: dochat/chroma_db")
else:
    print("Directory dochat/chroma_db already exists.")

Created directory: dochat/chroma_db


In [64]:
%%writefile dochat/main.py
#main.py — FastAPI app, lifespan, routes
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from contextlib import asynccontextmanager
import uuid, os, tempfile
import dochat.rag as rag   # your rag.py

# ── Lifespan: runs once on startup ────────────────────────────
# Initialises embedding model and Chroma ONCE when server starts,
# not on every request. Expensive models should never load per-request.
@asynccontextmanager
async def lifespan(app: FastAPI):
    rag.init_models()          # loads embedder + connects Chroma
    print("Models ready.")
    yield                      # server runs here
    print("Shutting down.")    # cleanup if needed

app = FastAPI(title="DocChat API", lifespan=lifespan)

# Allow your Next.js frontend to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],     # lock down to your frontend URL in prod
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Request/Response models ────────────────────────────────────
class QueryRequest(BaseModel):
    question: str
    doc_id:   str
    top_k:    int = 3

class UploadResponse(BaseModel):
    doc_id:      str
    chunk_count: int
    filename:    str

# ── Routes ────────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"message": "Welcome to the DocChat API! Visit /docs for OpenAPI spec."}

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "docs_indexed": rag.total_chunks(),
    }

@app.post("/upload", response_model=UploadResponse)
async def upload_pdf(file: UploadFile = File(...)):
    if not file.filename.endswith(".pdf"):
        raise HTTPException(400, "Only PDF files accepted.")

    # Save upload to a temp file (UploadFile is a stream, not a path)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    doc_id      = str(uuid.uuid4())          # unique ID for this doc
    chunk_count = rag.ingest_pdf(tmp_path, doc_id)
    os.unlink(tmp_path)                        # clean up temp file

    return UploadResponse(
        doc_id=doc_id,
        chunk_count=chunk_count,
        filename=file.filename
    )

@app.post("/query")
async def query_doc(req: QueryRequest):
    # StreamingResponse lets tokens reach the client as they generate
    # — critical for UX, nobody wants to stare at a spinner for 5s
    return StreamingResponse(
        rag.stream_answer(req.question, req.doc_id, req.top_k),
        media_type="text/event-stream"
    )

if __name__ == '__main__':
    # For local development, you might run this directly with uvicorn
    # In Colab, you'd typically use a tool like ngrok or run it as a module
    print('main.py is set up for FastAPI. Use uvicorn to run it or import it.')
    # uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting dochat/main.py


In [35]:
%%writefile dochat/rag.py
#rag.py — all RAG logic in one place
import fitz, os
import google.generativeai as genai
from sentence_transformers       import SentenceTransformer
from langchain_text_splitters    import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents    import Document
from langchain_core.embeddings   import Embeddings
from typing import List, AsyncIterator

# ── Module-level singletons ────────────────────────────────────
# These are None until init_models() is called at startup.
# Never instantiate heavy objects at import time.
_embedder:    SentenceTransformer = None
_vectorstore: Chroma              = None

CHROMA_DIR  = "./chroma_db"
COLLECTION  = "dochat_docs"
CHUNK_SIZE  = 300
OVERLAP     = 40

# ── Embedding wrapper (same as previous step) ─────────────────
class STEmbeddings(Embeddings):
    def __init__(self, model): self.model = model
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text])[0].tolist()

# ── Called once at server startup ─────────────────────────────
def init_models():
    global _embedder, _vectorstore
    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    _embedder    = SentenceTransformer("all-MiniLM-L6-v2")
    _vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=STEmbeddings(_embedder),
        collection_name=COLLECTION,
    )

# ── Ingest: PDF → chunks → Chroma ─────────────────────────────
def ingest_pdf(pdf_path: str, doc_id: str) -> int:
    text = "\n".join(
        p.get_text() for p in fitz.open(pdf_path)
    )
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=OVERLAP,
        separators=["\n\n", "\n", ". ", " "]
    )
    chunks = splitter.split_text(text)

    # Tag every chunk with doc_id for per-doc filtering at query time
    docs = [
        Document(
            page_content=c,
            metadata={"doc_id": doc_id, "chunk_idx": i}
        )
        for i, c in enumerate(chunks)
    ]
    _vectorstore.add_documents(docs)
    return len(docs)

# ── Query: retrieve + stream answer ───────────────────────────
async def stream_answer(
    question: str, doc_id: str, top_k: int = 3
) -> AsyncIterator[str]:

    # Filter retrieval to ONLY this user's document
    retriever = _vectorstore.as_retriever(
        search_kwargs={
            "k": top_k,
            "filter": {"doc_id": doc_id}   # ← multi-tenancy!
        }
    )
    relevant_docs = retriever.invoke(question)

    if not relevant_docs:
        yield "data: No relevant content found in this document.\n\n"
        return

    context = "\n\n---\n\n".join(d.page_content for d in relevant_docs)
    prompt  = f"""Answer using ONLY the context below.\nIf the answer is not present, say "I don't have that info."\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"""

    # Gemini streaming — yields chunks as they generate
    model    = genai.GenerativeModel("gemini-2.5-flash") # Changed model to gemini-2.5-flash
    response = model.generate_content(prompt, stream=True)

    for chunk in response:
        if chunk.text:
            # SSE format: "data: {text}\n\n"
            yield f"data: {chunk.text}\n\n"

# ── Utility ───────────────────────────────────────────────────
def total_chunks() -> int:
    return _vectorstore._collection.count()

if __name__ == '__main__':
    print('rag.py is intended to be imported as a module.')
    print('It defines functions to initialize the RAG system and create the RAG chain.')


Overwriting dochat/rag.py


In [26]:
%%writefile dochat/requirements.txt
pymupdf
sentence-transformers
langchain
langchain-text-splitters
langchain-community
langchain-core
chromadb
google-generativeai

Writing dochat/requirements.txt


In [36]:
#RUN IN COLAB — use this cell to start the server
!pip install fastapi uvicorn python-multipart -q

# Set your API key as env var before starting
import os
# Assuming GEMINI_API_KEY is stored in Colab secrets
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Start the server in the background
import subprocess, threading

def run_server():
    subprocess.run([
        "uvicorn", "dochat.main:app", # Specify the module path to main.py
        "--host", "0.0.0.0",
        "--port", "8000",
        "--reload" # Use --reload for development to pick up file changes
    ])

# Stop any existing server thread if it's still running
if 'thread' in locals() and thread.is_alive():
    print("Stopping existing server thread...")
    # A clean way to stop is often not straightforward with subprocess.run
    # For Colab, often re-running the cell is enough as it replaces the process

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server starting on port 8000...")
print("Please wait a few seconds for the server to fully initialize before testing again.")


Stopping existing server thread...
Server starting on port 8000...
Please wait a few seconds for the server to fully initialize before testing again.


In [41]:
import requests, json, time
time.sleep(3)   # wait for server to be ready

BASE = "http://localhost:8000"

# 1. Health check
resp = requests.get(f"{BASE}/health")
print("Health:", resp.json())

# 2. Upload a PDF
# IMPORTANT: Replace "your_file.pdf" with an actual PDF file path available in your Colab environment.
# For example, use "Rajkumar_Resume (1).pdf" if you uploaded it previously.

# First, check if the file exists to avoid an error
pdf_to_upload = "Rajkumar_Resume.pdf" # Or "Rajkumar_Resume (1).pdf"
import os
if not os.path.exists(pdf_to_upload):
    print(f"Error: PDF file '{pdf_to_upload}' not found. Please upload it or specify an existing file.")
else:
    with open(pdf_to_upload, "rb") as f:
        resp = requests.post(
            f"{BASE}/upload",
            files={"file": (pdf_to_upload, f, "application/pdf")}
        )
    upload_data = resp.json()
    print("Upload:", upload_data)
    doc_id = upload_data["doc_id"]    # save this!

    # 3. Query with streaming — reads SSE chunks as they arrive
    with requests.post(
        f"{BASE}/query",
        json={"question": "What are the skills of this candidate?", "doc_id": doc_id},
        stream=True
    ) as r:
        print("\nAnswer (streaming):")
        for line in r.iter_lines():
            if line and line.startswith(b"data: "):
                print(line[6:].decode(), end="", flush=True)
    print()
    print("If you see tokens printing one by one — your streaming pipeline is working end-to-end.")
    print("That's the same mechanism your Next.js frontend will consume.")

Health: {'status': 'ok', 'docs_indexed': 104}
Upload: {'doc_id': '8e06fdef-6a47-498e-8368-44ea0b345f59', 'chunk_count': 26, 'filename': 'Rajkumar_Resume.pdf'}

Answer (streaming):
The skills of this candidate are: Assessment & Mitigation (FMEA), Cross-Functional Leadership (40+ Members), MS Office Suite (Advanced), Microsoft Project (MSP), GMP, QMS, CQV, RCM, FMEA, EAM, HVAC, HPAPI, Cleanroom & Cold Chain Operations, Commissioning, Qualification & Validation (CQV), Reliability-Centered Maintenance (RCM), Vendor & Stakeholder Management, Budget & Scope Control (WBS), HPAPI & Cleanroom Operations, Safety Governance & GEMBA Leadership.
If you see tokens printing one by one — your streaming pipeline is working end-to-end.
That's the same mechanism your Next.js frontend will consume.


In [42]:
import os

# Install ngrok if not already installed
if not os.path.exists('/usr/local/bin/ngrok'):
    !curl -s -L https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip | funzip > /usr/local/bin/ngrok
    !chmod +x /usr/local/bin/ngrok

print("Ngrok setup complete.")

Ngrok setup complete.


In [61]:
#@markdown #### Enter your ngrok authtoken and run this cell to start the tunnel

!pip install pyngrok -q
from pyngrok import ngrok
import time

# Get a free ngrok token at ngrok.com (takes 1 minute)
# Then set it once.
# IMPORTANT: Replace "your-ngrok-token" with your actual ngrok authentication token.
NGROK_AUTH_TOKEN = "3Bd4Do0VdLEaMsaJ0dJFn1k03rX_7Hg7byHugXfADujDPddhT" #@param {type:"string"}

if NGROK_AUTH_TOKEN == "<YOUR_NGROK_AUTH_TOKEN>":
    print("Please replace '<YOUR_NGROK_AUTH_TOKEN>' with your actual ngrok authentication token.")
else:
    print("Ngrok authentication token set.")
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Terminate any existing ngrok tunnels to prevent conflicts
    ngrok.kill()

    # Start a new ngrok tunnel
    print("Starting ngrok tunnel...")
    try:
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
        print(f"Ngrok tunnel started. Public URL: {public_url}")

        # Store the public URL in a variable for later use
        global NGROK_PUBLIC_URL
        NGROK_PUBLIC_URL = public_url
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Please ensure your ngrok token is valid and you've run the ngrok executable installation cell.")


Ngrok authentication token set.
Starting ngrok tunnel...
Ngrok tunnel started. Public URL: https://overslack-partway-phung.ngrok-free.dev


In [62]:
#@markdown #### Enter your ngrok authtoken and run this cell to start the tunnel

!pip install pyngrok -q
from pyngrok import ngrok
import time

# Get a free ngrok token at ngrok.com (takes 1 minute)
# Then set it once.
# IMPORTANT: Replace "your-ngrok-token" with your actual ngrok authentication token.
NGROK_AUTH_TOKEN = "3Bd4Do0VdLEaMsaJ0dJFn1k03rX_7Hg7byHugXfADujDPddhT" #@param {type:"string"}

if NGROK_AUTH_TOKEN == "<YOUR_NGROK_AUTH_TOKEN>":
    print("Please replace '<YOUR_NGROK_AUTH_TOKEN>' with your actual ngrok authentication token.")
else:
    print("Ngrok authentication token set.")
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Terminate any existing ngrok tunnels to prevent conflicts
    ngrok.kill()

    # Start a new ngrok tunnel
    print("Starting ngrok tunnel...")
    try:
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
        print(f"Ngrok tunnel started. Public URL: {public_url}")

        # Store the public URL in a variable for later use
        global NGROK_PUBLIC_URL
        NGROK_PUBLIC_URL = public_url
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Please ensure your ngrok token is valid and you've run the ngrok executable installation cell.")

Ngrok authentication token set.
Starting ngrok tunnel...
Ngrok tunnel started. Public URL: https://overslack-partway-phung.ngrok-free.dev


In [63]:
import requests, json, time

time.sleep(3)   # Give the tunnel a moment to stabilize

# Use the public ngrok URL
BASE_URL = NGROK_PUBLIC_URL

print(f"Testing API endpoints using BASE_URL: {BASE_URL}")

# 1. Health check
try:
    resp = requests.get(f"{BASE_URL}/health")
    resp.raise_for_status() # Raise an exception for HTTP errors
    print("Health:", resp.json())
except requests.exceptions.RequestException as e:
    print(f"Error during health check: {e}")

# 2. Upload a PDF
# IMPORTANT: Replace "Rajkumar_Resume.pdf" with an actual PDF file path
# that is accessible in your Colab environment.
pdf_to_upload = "Rajkumar_Resume.pdf"

import os # Ensure os is imported for os.path.exists
if not os.path.exists(pdf_to_upload):
    print(f"Error: PDF file '{pdf_to_upload}' not found. Please ensure it's uploaded to Colab.")
else:
    try:
        with open(pdf_to_upload, "rb") as f:
            resp = requests.post(
                f"{BASE_URL}/upload",
                files={"file": (pdf_to_upload, f, "application/pdf")}
            )
        resp.raise_for_status() # Raise an exception for HTTP errors
        upload_data = resp.json()
        print("Upload:", upload_data)
        doc_id = upload_data["doc_id"]    # save this!

        # 3. Query with streaming — reads SSE chunks as they arrive
        print("\nAnswer (streaming) for 'What are the skills of this candidate?':")
        with requests.post(
            f"{BASE_URL}/query",
            json={"question": "What are the skills of this candidate?", "doc_id": doc_id},
            stream=True
        ) as r:
            r.raise_for_status() # Raise an exception for HTTP errors
            for line in r.iter_lines():
                if line and line.startswith(b"data: "):
                    print(line[6:].decode(), end="", flush=True)
        print()
        print("Streaming query complete. If you saw tokens printing one by one, your streaming pipeline is working end-to-end via ngrok.")

    except requests.exceptions.RequestException as e:
        print(f"Error during upload or query: {e}")

Testing API endpoints using BASE_URL: https://overslack-partway-phung.ngrok-free.dev
Health: {'status': 'ok', 'docs_indexed': 130}
Upload: {'doc_id': 'b22d5299-3a65-4512-bf20-8b190d17aae3', 'chunk_count': 26, 'filename': 'Rajkumar_Resume.pdf'}

Answer (streaming) for 'What are the skills of this candidate?':
The skills of this candidate are: Validation (CQV)Safety Governance & GEMBA Leadership
Streaming query complete. If you saw tokens printing one by one, your streaming pipeline is working end-to-end via ngrok.


In [65]:
#RUN IN COLAB — use this cell to start the server
!pip install fastapi uvicorn python-multipart -q

# Set your API key as env var before starting
import os
# Assuming GEMINI_API_KEY is stored in Colab secrets
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Start the server in the background
import subprocess, threading

def run_server():
    subprocess.run([
        "uvicorn", "dochat.main:app", # Specify the module path to main.py
        "--host", "0.0.0.0",
        "--port", "8000",
        "--reload" # Use --reload for development to pick up file changes
    ])

# Stop any existing server thread if it's still running
if 'thread' in locals() and thread.is_alive():
    print("Stopping existing server thread...")
    # A clean way to stop is often not straightforward with subprocess.run
    # For Colab, often re-running the cell is enough as it replaces the process

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server starting on port 8000...")
print("Please wait a few seconds for the server to fully initialize before testing again.")

Server starting on port 8000...
Please wait a few seconds for the server to fully initialize before testing again.
